# Ultimate NAKALA Workflow

Complete workflow for uploading, organizing, and curating data in NAKALA repository.
This notebook demonstrates all steps from initial upload to quality analysis.

**Note**: This notebook has been updated to work with the correct CSV column names and status values from o-nakala-core v3.0.0. The workflow_modules have been fixed to properly parse upload_results.csv and collections_output.csv files.

In [ ]:
# Setup and imports
import sys
import os
from pathlib import Path
import logging

# Add workflow modules to path
sys.path.append('workflow_modules')

# Import workflow modules
from workflow_config import WorkflowConfig
from data_uploader import DataUploader
from collection_manager import CollectionManager
from curator_operations import CuratorOperations
from metadata_enhancer import MetadataEnhancer
from quality_analyzer import QualityAnalyzer
from workflow_summary import WorkflowSummary

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("🚀 Ultimate NAKALA Workflow - All modules loaded successfully!")

## Step 1: Configuration Setup

Configure the workflow with API credentials and dataset paths.

In [ ]:
# Initialize workflow configuration
api_key = os.getenv('NAKALA_API_KEY', '33170cfe-f53c-550b-5fb6-4814ce981293')
base_path = '../sample_dataset'

# Create workflow configuration
workflow_config = WorkflowConfig(api_key=api_key, base_path=base_path)
config = workflow_config.get_config_dict()

# Add additional configuration parameters
config.update({
    'mode': 'folder',
    'batch_size': 10,
    'max_retries': 3,
    'timeout': 300
})

print("📋 Workflow Configuration:")
print(f"   API URL: {config['api_url']}")
print(f"   Base Path: {config['base_path']}")
print(f"   Dataset CSV: {config['dataset_csv']}")
print(f"   Collections CSV: {config['collections_csv']}")
print(f"   Mode: {config['mode']}")
print(f"   Batch Size: {config['batch_size']}")

## Step 2: Data Upload

Upload datasets to NAKALA repository using the configured parameters.

In [ ]:
# Initialize data uploader
uploader = DataUploader(config)

print("📤 Starting data upload process...")

# Perform upload operation
try:
    upload_results = uploader.upload_datasets(
        mode=config['mode']  # Use folder mode from configuration
    )
    
    print(f"✅ Upload completed successfully!")
    print(f"   Items uploaded: {upload_results['stats']['successful_uploads']}")
    print(f"   Failed uploads: {upload_results['stats']['failed_uploads']}")
    print(f"   Results file: {upload_results['stats']['results_file']}")
    
except Exception as e:
    print(f"❌ Upload failed: {e}")
    upload_results = None

## Step 3: Collection Management

Create and organize collections based on uploaded data.

In [ ]:
# Initialize collection manager
collection_manager = CollectionManager(config)

print("📁 Starting collection management...")

# Create collections from upload results
if upload_results and upload_results['success']:
    try:
        collection_results = collection_manager.create_collections(
            upload_results_file=upload_results['stats']['results_file']
        )
        
        print(f"✅ Collection creation completed!")
        print(f"   Collections created: {collection_results['stats']['successful_collections']}")
        print(f"   Failed collections: {collection_results['stats']['failed_collections']}")
        
    except Exception as e:
        print(f"❌ Collection creation failed: {e}")
        collection_results = None
else:
    print("⏭️  Skipping collection creation - no upload results available")
    collection_results = None

## Step 4: Metadata Enhancement

Enhance and standardize metadata for uploaded items.

In [ ]:
# Initialize metadata enhancer
metadata_enhancer = MetadataEnhancer(config)

print("🔍 Starting metadata enhancement...")

# Enhance metadata for uploaded items
try:
    # Generate data enhancements
    data_enhancement_results = metadata_enhancer.generate_data_enhancements(
        upload_results_file=upload_results['stats']['results_file'] if upload_results else None
    )
    
    # Generate collection enhancements if collections were created
    collection_enhancement_results = None
    if collection_results and collection_results.get('success'):
        collection_enhancement_results = metadata_enhancer.generate_collection_enhancements()
    
    enhancement_results = {
        'data_enhancements': data_enhancement_results,
        'collection_enhancements': collection_enhancement_results,
        'success': True
    }
    
    print(f"✅ Metadata enhancement completed!")
    print(f"   Data enhancements: {data_enhancement_results.get('stats', {}).get('enhancements_generated', 0)}")
    if collection_enhancement_results:
        print(f"   Collection enhancements: {collection_enhancement_results.get('stats', {}).get('enhancements_generated', 0)}")
    
except Exception as e:
    print(f"❌ Metadata enhancement failed: {e}")
    enhancement_results = None

## Step 5: Curator Operations

Perform advanced curation operations including validation and duplicate detection.

In [ ]:
# Initialize curator operations
curator = CuratorOperations(config)

print("🔧 Starting curator operations...")

# Perform comprehensive curation
try:
    # Apply all curation operations (datasets and collections)
    curation_results = curator.curate_all()
    
    print(f"✅ Curator operations completed!")
    
    # Display dataset curation results
    if curation_results['dataset_curation']:
        dataset_stats = curation_results['dataset_curation']['stats']
        print(f"   Dataset modifications: {dataset_stats.get('modifications_applied', 0)}")
    
    # Display collection curation results
    if curation_results['collection_curation']:
        collection_stats = curation_results['collection_curation']['stats']
        print(f"   Collection modifications: {collection_stats.get('modifications_applied', 0)}")
    
    validation_results = curation_results
    duplicate_results = None  # Not available in this workflow
    
except Exception as e:
    print(f"❌ Curator operations failed: {e}")
    validation_results = None
    duplicate_results = None

## Step 6: Quality Analysis

Generate comprehensive quality reports and analysis.

In [ ]:
# Initialize quality analyzer
quality_analyzer = QualityAnalyzer(config)

print("📊 Starting quality analysis...")

# Generate quality report
try:
    quality_results = quality_analyzer.generate_quality_report(
        scope='all',  # Analyze all items and collections
        output_file=None  # Use default output file
    )
    
    print(f"✅ Quality analysis completed!")
    
    # Analyze quality trends
    trends = quality_analyzer.analyze_quality_trends()
    if trends and isinstance(trends, dict):
        print(f"\n📈 Quality Trends:")
        print(f"   Metadata Completeness: {trends.get('metadata_completeness', 'Unknown')}")
        print(f"   Common Issues: {len(trends.get('common_issues', []))}")
        print(f"   Improvement Areas: {len(trends.get('improvement_areas', []))}")
    
    # Export quality summary
    summary_file = quality_analyzer.export_quality_summary()
    print(f"   Quality summary exported to: {summary_file}")
    
except Exception as e:
    print(f"❌ Quality analysis failed: {e}")
    quality_results = None

## Step 7: Workflow Summary

Generate comprehensive workflow summary and final report.

In [ ]:
# Initialize workflow summary
workflow_summary = WorkflowSummary(config)

print("📋 Generating workflow summary...")

# Generate comprehensive summary
try:
    summary_results = workflow_summary.generate_comprehensive_summary()
    
    print(f"✅ Workflow summary generated!")
    
    # Display final statistics
    print(f"\n🎯 Final Workflow Statistics:")
    overall_stats = summary_results.get('overall_statistics', {})
    print(f"   Total Operations: {overall_stats.get('total_operations', 'Unknown')}")
    print(f"   Successful Operations: {overall_stats.get('successful_operations', 'Unknown')}")
    print(f"   Items Created: {overall_stats.get('total_items_created', 'Unknown')}")
    print(f"   Enhancements Applied: {overall_stats.get('total_enhancements_applied', 'Unknown')}")
    print(f"   Success Rate: {overall_stats.get('workflow_success_rate', 0):.1f}%")
    
    # Display operation summaries
    if summary_results.get('upload_summary', {}).get('available'):
        upload_stats = summary_results['upload_summary']
        print(f"   Upload: {upload_stats.get('total_datasets', 0)} datasets")
    
    if summary_results.get('collections_summary', {}).get('available'):
        collection_stats = summary_results['collections_summary']
        print(f"   Collections: {collection_stats.get('total_collections', 0)} collections")
    
    if summary_results.get('quality_summary', {}).get('available'):
        quality_stats = summary_results['quality_summary']
        print(f"   Quality Score: {quality_stats.get('overall_quality_score', 0):.1f}")
    
except Exception as e:
    print(f"❌ Workflow summary failed: {e}")
    summary_results = None

## Workflow Completion

The ultimate NAKALA workflow has been completed. All steps from upload to quality analysis have been executed.

In [ ]:
print("\n🎉 Ultimate NAKALA Workflow Completed!")
print("=" * 60)

# Final verification
verification_steps = [
    ("Configuration", config is not None),
    ("Upload", upload_results is not None and upload_results.get('success', False)),
    ("Collections", collection_results is not None),
    ("Metadata Enhancement", enhancement_results is not None),
    ("Validation", validation_results is not None),
    ("Quality Analysis", quality_results is not None),
    ("Workflow Summary", summary_results is not None)
]

print("🔍 Workflow Step Verification:")
for step_name, success in verification_steps:
    status = "✅ COMPLETED" if success else "❌ FAILED"
    print(f"   {step_name}: {status}")

print("=" * 60)
print("📚 Check the following files for detailed results:")
print("   - upload_results.csv (upload details)")
print("   - collections_output.csv (collection details)")
print("   - quality_report.json (quality analysis)")
print("   - workflow_summary.json (comprehensive summary)")